# ZenodoClient Sandbox Demo

This notebook demonstrates the new `ZenodoClient` against the Zenodo sandbox. It covers typed metadata authoring, draft creation, bucket uploads, raw metadata fields, and safe cleanup.

The sandbox URL is intentionally explicit in the notebook. Only the token is loaded from the environment.

## Environment

Add this entry to `secrets/.env` and start Jupyter from a shell that has sourced it:

```bash
export ZENODO_SANDBOX_TOKEN="..."
```

The token must come from `https://sandbox.zenodo.org`, not production Zenodo. Use `deposit:write` for draft/upload/update. Add `deposit:actions` only if you want to publish or create new versions.

In [1]:
from datetime import date
from pathlib import Path


def require_env(name: str) -> str:
    import os
    value = os.getenv(name)
    if value is None or not value.strip():
        raise RuntimeError(f"Required environment variable {name} is not set.")
    return value.strip()


sandbox_url = "https://sandbox.zenodo.org"
token = require_env("ZENODO_SANDBOX_TOKEN")

In [2]:
from courier import ZenodoClient
from courier.services.zenodo import (
    Contributor,
    Creator,
    RelatedIdentifier,
    ZenodoMetadata,
)

client = ZenodoClient(address=sandbox_url, token=token)
client.base_url

'https://sandbox.zenodo.org'

## Create A Small Artifact

In [3]:
artifact = Path("demo_artifact.txt")
artifact.write_text("Hello from courier's Zenodo sandbox demo.\n", encoding="utf-8")
artifact

PosixPath('demo_artifact.txt')

## Build Minimal Typed Metadata

`ZenodoMetadata` is an authoring object. This first metadata object is intentionally minimal and should be accepted by the sandbox with a normal `deposit:write` token. Optional identifiers, communities, and grants are shown later because they may be rejected if the values are not valid in the sandbox.


In [4]:
metadata = ZenodoMetadata.software()
metadata.title = "courier Zenodo sandbox demo"
metadata.publication_date = date.today()
metadata.description = "Small demonstration upload created with courier."
metadata.license = "cc-by-4.0"
metadata.version = "0.1.0"
metadata.language = "eng"

metadata.creators.append(
    Creator(
        family_name="Doe",
        given_names="Jane",
        affiliation="Example Institute",
    )
)

metadata.keywords.extend(["courier", "zenodo", "sandbox"])

In [5]:
metadata.to_api_dict()

{'upload_type': 'software',
 'publication_date': '2026-05-13',
 'title': 'courier Zenodo sandbox demo',
 'creators': [{'name': 'Doe, Jane', 'affiliation': 'Example Institute'}],
 'description': 'Small demonstration upload created with courier.',
 'access_right': 'open',
 'license': 'cc-by-4.0',
 'keywords': ['courier', 'zenodo', 'sandbox'],
 'version': '0.1.0',
 'language': 'eng'}

In [6]:
metadata.to_payload()

{'metadata': {'upload_type': 'software',
  'publication_date': '2026-05-13',
  'title': 'courier Zenodo sandbox demo',
  'creators': [{'name': 'Doe, Jane', 'affiliation': 'Example Institute'}],
  'description': 'Small demonstration upload created with courier.',
  'access_right': 'open',
  'license': 'cc-by-4.0',
  'keywords': ['courier', 'zenodo', 'sandbox'],
  'version': '0.1.0',
  'language': 'eng'}}

## Create A Draft, Upload The File, And Set Metadata

This keeps the record unpublished. Publishing is shown later as an explicit optional step.

In [7]:
draft = client.depositions.create(prereserve_doi=True)
draft.id

499007

In [8]:
uploaded = client.files.upload(draft, artifact)
[(file.filename, file.size) for file in uploaded]

[('demo_artifact.txt', 42)]

In [9]:
draft = client.depositions.set_metadata(draft, metadata)
draft.links.html

'https://sandbox.zenodo.org/deposit/499007'

## Optional Metadata Enrichment

The fields below demonstrate more of the metadata model, but they are intentionally not part of the first successful sandbox update. ORCID values must be real ORCIDs, community identifiers must exist in the sandbox, and grant identifiers must match values Zenodo accepts. Use this section after the minimal draft update works.


In [10]:
enriched_metadata = ZenodoMetadata.from_dict(metadata.to_api_dict())

# Use a real ORCID before uncommenting.
# enriched_metadata.creators[0].orcid = "0000-0002-1825-0097"

enriched_metadata.contributors.append(
    Contributor(
        name="Example Group",
        type="ResearchGroup",
        affiliation="Example Institute",
    )
)

enriched_metadata.related_identifiers.append(
    RelatedIdentifier(
        identifier="https://github.com/pyiron/courier",
        relation="isSupplementTo",
        resource_type="software",
    )
)

# Only use communities/grants that are valid in your sandbox account.
# from courier.services.zenodo import CommunityRef, GrantRef
# enriched_metadata.communities.append(CommunityRef(identifier="your-sandbox-community"))
# enriched_metadata.grants.append(GrantRef(id="10.13039/501100000000::12345"))

enriched_metadata.to_api_dict()

{'upload_type': 'software',
 'publication_date': '2026-05-13',
 'title': 'courier Zenodo sandbox demo',
 'creators': [{'name': 'Doe, Jane', 'affiliation': 'Example Institute'}],
 'description': 'Small demonstration upload created with courier.',
 'access_right': 'open',
 'license': 'cc-by-4.0',
 'keywords': ['courier', 'zenodo', 'sandbox'],
 'related_identifiers': [{'identifier': 'https://github.com/pyiron/courier',
   'relation': 'isSupplementTo',
   'resource_type': 'software'}],
 'contributors': [{'name': 'Example Group',
   'type': 'ResearchGroup',
   'affiliation': 'Example Institute'}],
 'version': '0.1.0',
 'language': 'eng'}

In [11]:
# Run this only after inspecting enriched_metadata.to_api_dict().
# from courier.services.zenodo import ZenodoApiError
# try:
#     draft = client.depositions.set_metadata(draft, enriched_metadata)
# except ZenodoApiError as exc:
#     print(exc)
#     for error in exc.errors or []:
#         print(error.field, error.message)
# else:
#     draft.links.html

## Raw Metadata For Fields Not Yet Modeled

`ZenodoMetadata` covers the common authoring path. For documented Zenodo fields that are not modeled yet, pass a raw metadata dictionary. The client wraps it as `{"metadata": ...}` but does not run local metadata validation on raw mappings.

`references` is a useful example of a Zenodo metadata field not currently modeled by `ZenodoMetadata`.

In [12]:
raw_metadata = metadata.to_api_dict()
raw_metadata["references"] = [
    "Doe J. (2026). Example reference for a sandbox upload. DOI:10.0000/example",
    "Smith A. (2025). Another reference. Example Journal.",
]

draft = client.depositions.set_metadata(draft, raw_metadata)
draft.metadata.get("references")

['Doe J. (2026). Example reference for a sandbox upload. DOI:10.0000/example',
 'Smith A. (2025). Another reference. Example Journal.']

In [13]:
raw_only_metadata = {
    "upload_type": "dataset",
    "publication_date": date.today().isoformat(),
    "title": "courier raw metadata sandbox demo",
    "creators": [{"name": "Doe, Jane", "affiliation": "Example Institute"}],
    "description": "A draft using a raw metadata mapping.",
    "access_right": "open",
    "license": "cc-by-4.0",
    "references": ["Doe J. (2026). Raw metadata reference. DOI:10.0000/raw-example"],
}

# client.depositions.set_metadata(draft, raw_only_metadata)
raw_only_metadata

{'upload_type': 'dataset',
 'publication_date': '2026-05-13',
 'title': 'courier raw metadata sandbox demo',
 'creators': [{'name': 'Doe, Jane', 'affiliation': 'Example Institute'}],
 'description': 'A draft using a raw metadata mapping.',
 'access_right': 'open',
 'license': 'cc-by-4.0',
 'references': ['Doe J. (2026). Raw metadata reference. DOI:10.0000/raw-example']}

## Round-Trip Metadata Payloads

`from_dict()` is for examples and local payload manipulation. Server responses still use `DepositionInfo`, whose `metadata` field remains a raw dictionary.

In [14]:
round_tripped = ZenodoMetadata.from_dict({"metadata": raw_metadata})
round_tripped.title, round_tripped.creators[0].name

('courier Zenodo sandbox demo', 'Doe, Jane')

## Optional Publish

Publishing requires the `deposit:actions` scope. Leave this commented out for routine demos.

In [15]:
# published = client.depositions.publish(draft)
# published.links.html

## Cleanup

Unpublished sandbox drafts can be deleted. Published sandbox records should be treated as persistent test records.

In [16]:
client.depositions.delete(draft)
artifact.unlink(missing_ok=True)

## Extending `ZenodoMetadata`

Add a typed metadata field when callers should not have to use the raw dictionary escape hatch:

1. Add the dataclass attribute to `ZenodoMetadata`.
2. Add serialization in `to_api_dict()`.
3. Add parsing in `from_dict()`.
4. Add focused metadata tests.

For a list-valued field like `references`, add `references: list[str] = field(default_factory=list)`, serialize it only when non-empty, parse it with a helper like `_string_list(...)`, and test both serialization and parsing.